In [6]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
# Reading the data and converting into dataframe
df = pd.read_csv("data/train.csv")

In [7]:
# Printing first 5 rows
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Profiling the data

### shape, column types, and missingness per column.

In [10]:
# Shape
df.shape

(891, 12)

In [13]:
# Column details
list(df.columns) 
# used list for better visualization

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

# Data Type of each column


In [16]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [17]:
# No need to change the datatype of any column

In [15]:
# number of rows,column names, non-null values, data types, memory usage by info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [18]:
# interpretation
# Age :- In age , 891 - 714 = 177 enteries are missing , which is 19.9% of data we can't remove the 19.9% of data.That's why we should impute the age by median.
# Cabin :- In Cabin column,  891 - 204 = 687 enteries are missing , which is 77.1% of data of cabin ,Filling that much of data by median or mode will be a stupid idea , so we will drop the column cabin.
# Embarked : 2 enteries are missing,The two missing Embarked values should be replaced with the mode because the missing proportion is negligible and Embarked is categorical.

In [19]:
# Checking the duplicates
df.duplicated().sum()

np.int64(0)

In [20]:
# Means duplicated row found = 0

### Outliers

In [21]:
# Basic stats of numerical columns
df[["Age", "Fare", "SibSp", "Parch"]].describe()

,Age,Fare,SibSp,Parch
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,32.204208,0.523008,0.381594
std,14.526497,49.693429,1.102743,0.806057
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,7.910400,0.000000,0.000000
50%,28.000000,14.454200,0.000000,0.000000
75%,38.000000,31.000000,1.000000,0.000000
max,80.000000,512.329200,8.000000,6.000000


In [22]:
# calculating outlier for fare
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

fare_outliers = df[
    (df["Fare"] < lower_bound) |
    (df["Fare"] > upper_bound)
]

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Number of outliers:", len(fare_outliers))

Lower Bound: -26.724
Upper Bound: 65.6344
Number of outliers: 116


In [23]:
# Analysing the outliers
fare_outliers["Fare"].sort_values()

151     66.6000
336     66.6000
369     69.3000
641     69.3000
201     69.5500
         ...   
341    263.0000
438    263.0000
258    512.3292
679    512.3292
737    512.3292
Name: Fare, Length: 116, dtype: float64

In [26]:
df[df["Fare"] > 263][
    ["PassengerId", "Survived", "Pclass", "Name", "Ticket", "Fare"]
].sort_values("Fare")

,PassengerId,Survived,Pclass,Name,Ticket,Fare
258,259,1,1,"Ward, Miss. Anna",PC 17755,512.3292
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",PC 17755,512.3292
737,738,1,1,"Lesurer, Mr. Gustave J",PC 17755,512.3292


In [ ]:
# Form this we should not remove the outliers in fare like "512.3292" because this can be a genuine fair price , ticket "PC 17755" has same fare for all enteries

In [27]:
# Cheking if a ticket having different values
ticket_fare_check = (
    df.groupby("Ticket")["Fare"]
      .nunique()
      .sort_values(ascending=False)
)

ticket_fare_check.head(20)

Ticket
7534                 2
367655               1
368323               1
36864                1
36865                1
36866                1
368703               1
36928                1
36947                1
36963                1
36967                1
STON/O 2. 3101293    1
370129               1
370365               1
370369               1
370370               1
370371               1
370372               1
370373               1
370375               1
Name: Fare, dtype: int64

In [30]:
# Ticket = "7534" has two prices, identifying further
df[df["Ticket"] == "7534"][
    ["PassengerId", "Name", "Ticket", "Fare", "Pclass", "Sex", "Age"]
]


,PassengerId,Name,Ticket,Fare,Pclass,Sex,Age
138,139,"Osen, Mr. Olaf Elon",7534,9.2167,3,male,16.0
876,877,"Gustafsson, Mr. Alfred Ossian",7534,9.8458,3,male,20.0


In [31]:
#The Ticket field is not perfectly consistent with Fare, as ticket 7534 appears with two different fare values. This inconsistency should be investigated before using Ticket-Fare relationships for modeling or feature engineering.

In [35]:
# Handling outliers for SibSp
# IQR
Q1 = 0
Q3 = 1
IQR = Q3 - Q1
# Upper bound
Upper_Bound = Q3 + 1.5 * IQR
Lower_Bound = Q1 - 1.5 * IQR
print(Upper_Bound)
print(Lower_Bound)

2.5
-1.5


In [36]:
sibsp_outliers = df[df["SibSp"] > 2.5]

print("Number of SibSp outliers:", len(sibsp_outliers))
print(sibsp_outliers["SibSp"].value_counts().sort_index())

Number of SibSp outliers: 46
SibSp
3    16
4    18
5     5
8     7
Name: count, dtype: int64


In [38]:
# Handling outliers for SibSp
# IQR
Q1 = 0
Q3 = 0
IQR = Q3 - Q1
# Upper bound
Upper_Bound = Q3 + 1.5 * IQR
Lower_Bound = Q1 - 1.5 * IQR
print(Upper_Bound)
print(Lower_Bound)

0.0
0.0


In [39]:
# Check the frequency of each Parch value
df["Parch"].value_counts().sort_index()

Parch
0    678
1    118
2     80
3      5
4      4
5      5
6      1
Name: count, dtype: int64

In [40]:
# Parch was treated as a discrete relationship-count variable rather than a continuous variable, so IQR-based outlier removal was not applied. Its observed values were retained because they represent plausible family relationships.

## Actual problems in dataset